In [28]:
# Sam Brown
# sam_brown@mines.edu
# June 27 2025
# Goal: Use a regular neural network with our new features (tidal amplitudes between events) to predict slip size

import sys
sys.path.append("/Users/sambrown04/Documents/SURF/whillans-surf/notebooks/SURF")

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

df = pd.read_csv("/Users/sambrown04/Documents/SURF/Preproc_data/10-18.csv")
df = df[604:3001] # Adjust for missing data areas if needed

In [36]:
# Will do a little bit of preprocessing to see how including more in depth H,L 
sequence = [1 if x == 1 else -1 for x in df['high_t_evt']]

df['cumult'] = np.cumsum(sequence)
df['hpastone'] = df['high_t_evt'].shift(1)
df['hpasttwo'] = df['high_t_evt'].shift(2)
df['hpastthree'] = df['high_t_evt'].shift(3)

df =  df[610:3001]

In [38]:
df.head(10)

,tide_deriv,form_fac,time_since,slip_size_standardized,high_t_evt,start_time,sev_stds,pre-s_stds,tide_height,inter_form_Fac,A_diurn,A_semidiurn,cumult,hpastone,hpasttwo,hpastthree
1214,-0.173653,0.770640,880.0,-0.708618,0,2012-03-19 10:15:00,-0.099609,0.370271,-4.225656,0.878199,-39.840697,-45.366345,153,0.0,1.0,0.0
1215,-0.356734,0.770640,685.0,-1.083245,0,2012-03-19 21:40:00,-0.233474,-0.264301,-50.121969,0.749932,31.505201,-42.010733,152,0.0,0.0,1.0
1216,-0.264323,0.501299,1285.0,1.286292,1,2012-03-20 19:05:00,0.294528,0.344374,29.902842,0.534803,-23.404607,-43.763063,153,0.0,0.0,0.0
1217,-0.193084,0.232693,755.0,-0.199085,1,2012-03-21 07:40:00,-0.640188,-0.141270,24.849135,0.448594,-17.314389,-38.596991,154,1.0,0.0,0.0
1218,-0.279763,0.232693,765.0,-0.440268,1,2012-03-21 20:25:00,-0.464205,0.132378,17.917613,0.137525,4.964660,-36.100118,155,1.0,1.0,0.0
1219,-0.273179,0.326250,750.0,-0.661521,1,2012-03-22 08:55:00,-0.589024,0.127969,2.302367,0.231997,-7.889795,-34.008140,156,1.0,1.0,1.0
1220,-0.189686,0.326250,830.0,-0.828921,0,2012-03-22 22:45:00,-0.295163,-0.004790,-2.815216,0.351947,-10.416523,-29.596880,155,1.0,1.0,1.0
1221,0.101564,1.844238,1760.0,1.708015,1,2012-03-24 04:05:00,2.563472,0.090616,18.829938,0.818509,20.034110,-24.476361,156,0.0,1.0,1.0
1222,0.052726,4.883429,1385.0,1.530759,1,2012-03-25 03:10:00,1.725659,-0.114912,28.497400,2.399330,-33.750880,14.066793,157,1.0,0.0,1.0
1223,0.005935,10.666290,1470.0,1.966480,1,2012-03-26 03:40:00,1.445909,0.355986,45.427469,7.437381,41.483161,5.577657,158,1.0,1.0,0.0


In [42]:
# Features and target
X = df[['tide_deriv', 'time_since', 'high_t_evt', 'tide_height', 'A_diurn', 'A_semidiurn', 'sev_stds', 'cumult', 'hpastone', 'hpasttwo', 'hpastthree']]
y =df['slip_size_standardized'].values.reshape(-1,1)
 
# Split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2, random_state = 42)

# Standardize 
X_scaler = StandardScaler()
X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled = y_scaler.transform(y_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

In [44]:
class Net(nn.Module):
    def __init__(self):
        super(Net,self).__init__()
        self.fc1 = nn.Linear(11, 55)
        self.fc2 = nn.Linear(55, 21)
        self.fc3 = nn.Linear(21,11)
        self.output = nn.Linear(11,1)
        
    def forward(self,x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.output(x) # Regressing, no activation
        return x

In [76]:
model = Net()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = .01)


In [78]:
# Train
epochs = 200
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad() # Clears grad

    # predictions and loss
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    #Backprop
    loss.backward()

    #Update
    optimizer.step()

    if (epoch+1) % 20 == 0: # Print update to ensure no problems
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [20/200], Loss: 0.1795
Epoch [40/200], Loss: 0.1094
Epoch [60/200], Loss: 0.0749
Epoch [80/200], Loss: 0.0589
Epoch [100/200], Loss: 0.0511
Epoch [120/200], Loss: 0.0455
Epoch [140/200], Loss: 0.0408
Epoch [160/200], Loss: 0.0374
Epoch [180/200], Loss: 0.0345
Epoch [200/200], Loss: 0.0340


In [80]:
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_tensor)
    y_pred = y_scaler.inverse_transform(y_pred_scaled.detach().numpy())
    y_test_orig = y_scaler.inverse_transform(y_test_tensor.detach().numpy())
r2 = r2_score(y_test_orig, y_pred)
print(r2)


0.9129538536071777
